<a href="https://colab.research.google.com/github/vlazacck/Ethiomart/blob/task345/modelt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets
from datasets import load_dataset

# Define dataset path
data_files = {"train": "/content/amharic_data_conll_subset.txt"}

# Load dataset as a text file
raw_datasets = load_dataset("text", data_files=data_files)

# Rename the dataset to 'train'
raw_datasets = raw_datasets.rename_column("text", "train")

# Inspect the dataset
print(raw_datasets)
print(raw_datasets["train"][0])  # Check the first example

In [7]:
!pip install datasets
from datasets import load_dataset
from transformers import AutoTokenizer
import re

# Define dataset path
data_files = {"train": "/content/amharic_data_conll_subset.txt"}

# Load dataset as a text file
raw_datasets = load_dataset("text", data_files=data_files)

# Rename the dataset to 'train'
raw_datasets = raw_datasets.rename_column("text", "train")

# Inspect the dataset
print(raw_datasets)
print(raw_datasets["train"][0])  # Check the first example


# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# Parsing text into tokens and labels
def parse_text(example):
    tokens, labels = [], []
    # Access the renamed column 'train' instead of 'text'
    for line in example["train"].split("\n"):
        if line.strip():
            token, label = line.split("\t")
            tokens.append(token)
            labels.append(label)
    return {"tokens": tokens, "labels": labels}

# Apply parsing
parsed_datasets = raw_datasets.map(parse_text)

# ... (rest of the code in cell 5)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['train'],
        num_rows: 1611
    })
})
{'train': 'ዕቃዎችን\tB-PRODUCT'}


Map:   0%|          | 0/1611 [00:00<?, ? examples/s]

In [9]:
from transformers import AutoTokenizer
import re

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# Parsing text into tokens and labels
def parse_text(example):
    tokens, labels = [], []
    # Access the renamed column 'train' instead of 'text'
    for line in example["train"].split("\n"):  # Change 'text' to 'train'
        if line.strip():
            token, label = line.split("\t")
            tokens.append(token)
            labels.append(label)
    return {"tokens": tokens, "labels": labels}

# Apply parsing
parsed_datasets = raw_datasets.map(parse_text)

# Convert labels to integers
label_to_id = {"O": 0, "B-PRODUCT": 1, "I-PRODUCT": 2, "B-LOC": 3, "I-LOC": 4, "B-PRICE": 5, "I-PRICE": 6}
id_to_label = {v: k for k, v in label_to_id.items()}

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )
    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = [-100 if word_id is None else label_to_id[label[word_id]] for word_id in word_ids]
        labels.append(aligned_labels)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply tokenization
tokenized_datasets = parsed_datasets.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/1611 [00:00<?, ? examples/s]

In [10]:
def parse_text(example):
    tokens, labels = [], []
    for line in example["text"].split("\n"):  # Error occurs here
        if line.strip():
            token, label = line.split("\t")
            tokens.append(token)
            labels.append(label)
    return {"tokens": tokens, "labels": labels}

In [11]:
from transformers import DataCollatorForTokenClassification

# Create a data collator for token classification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, max_length=128)

# Verify data collator output
# Convert the Dataset slice to a list of dictionaries,
# but ensure you are only selecting the required features for the collator
features = [{k: v for k, v in tokenized_datasets["train"][i].items() if k in ["input_ids", "attention_mask", "labels"]} for i in range(4)]
batch = data_collator(features)
print(batch)

{'input_ids': tensor([[    0, 35587,  6550, 36415,     2,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,  

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2691: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [12]:
from transformers import AutoModelForTokenClassification

num_labels = len(label_to_id)
model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base", num_labels=num_labels)


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
from sklearn.metrics import classification_report
import numpy as np

# Define a metric computation function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id_to_label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = classification_report(
        true_labels, true_predictions, zero_division=0, output_dict=True
    )
    return {
        "precision": results["weighted avg"]["precision"],
        "recall": results["weighted avg"]["recall"],
        "f1": results["weighted avg"]["f1-score"],
    }


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    save_total_limit=2,
    load_best_model_at_end=True,
    save_strategy="epoch",  # Save at the end of each epoch to match evaluation strategy
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [24]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


<ipython-input-24-5cf55639dd3c>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [23]:
!pip install datasets transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from sklearn.metrics import classification_report


# 1. Load and preprocess data
data_files = {"train": "/content/amharic_data_conll_subset.txt"}
raw_datasets = load_dataset("text", data_files=data_files)
raw_datasets = raw_datasets.rename_column("text", "train")
raw_datasets = raw_datasets["train"].train_test_split(test_size=0.2, seed=42)
raw_datasets['validation'] = raw_datasets.pop('test')


# 2. Parsing text into tokens and labels
def parse_text(example):
    tokens, labels = [], []
    for line in example["train"].split("\n"):
        if line.strip():
            token, label = line.split("\t")
            tokens.append(token)
            labels.append(label)
    return {"tokens": tokens, "labels": labels}

parsed_datasets = raw_datasets.map(parse_text)


# 3. Tokenization and label alignment
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
label_to_id = {"O": 0, "B-PRODUCT": 1, "I-PRODUCT": 2, "B-LOC": 3, "I-LOC": 4, "B-PRICE": 5, "I-PRICE": 6}
id_to_label = {v: k for k, v in label_to_id.items()}

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )
    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = [-100 if word_id is None else label_to_id[label[word_id]] for word_id in word_ids]
        labels.append(aligned_labels)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = parsed_datasets.map(tokenize_and_align_labels, batched=True)


# 4. Define data collator and model
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, max_length=128)
num_labels = len(label_to_id)
model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base", num_labels=num_labels)


# 5. Define metric computation function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id_to_label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = classification_report(
        true_labels, true_predictions, zero_division=0, output_dict=True
    )
    return {
        "precision": results["weighted avg"]["precision"],
        "recall": results["weighted avg"]["recall"],
        "f1": results["weighted avg"]["f1-score"],
    }


# 6. Define training arguments and trainer
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    save_total_limit=2,
    load_best_model_at_end=True,
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Map:   0%|          | 0/1288 [00:00<?, ? examples/s]

Map:   0%|          | 0/323 [00:00<?, ? examples/s]

Map:   0%|          | 0/1288 [00:00<?, ? examples/s]

Map:   0%|          | 0/323 [00:00<?, ? examples/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-23-2d256d46cb93>:98: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [21]:
!pip install datasets
from datasets import load_dataset

# Define dataset path
data_files = {"train": "/content/amharic_data_conll_subset.txt"}

# Load dataset as a text file
raw_datasets = load_dataset("text", data_files=data_files)

# Rename the dataset to 'train'
raw_datasets = raw_datasets.rename_column("text", "train")

# Split the dataset into train and validation sets (e.g., 80% train, 20% validation)
# Use the 'test_size' parameter to define the validation set size,
# and directly name the validation split using 'train_test_split(test_size=0.2, seed=42)'
raw_datasets = raw_datasets["train"].train_test_split(test_size=0.2, seed=42)

# Rename the 'test' split to 'validation'
# Instead of using rename_column, which is for column names,
# we use pop to remove the 'test' split and assign it to 'validation'
raw_datasets['validation'] = raw_datasets.pop('test')

# Inspect the dataset
print(raw_datasets)
print(raw_datasets["train"][0])  # Check the first example in the training set
print(raw_datasets["validation"][0])  # Check the first example in the validation set

DatasetDict({
    train: Dataset({
        features: ['train'],
        num_rows: 1288
    })
    validation: Dataset({
        features: ['train'],
        num_rows: 323
    })
})
{'train': 'ዝውውርን\tO'}
{'train': 'ጥያቄ\tO'}


In [33]:
# Get predictions from the trained model
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"]) # Assumes 'validation' is your evaluation split

# Extract predicted labels and true labels
true_predictions = [
    [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(np.argmax(predictions, axis=2), labels)
]
true_labels = [
    [id_to_label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(np.argmax(predictions, axis=2), labels)
]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2691: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss


ValueError: You appear to be using a legacy multi-label data representation. Sequence of sequences are no longer supported; use a binary array or sparse matrix instead - the MultiLabelBinarizer transformer can convert to this format.

In [32]:
from sklearn.preprocessing import MultiLabelBinarizer

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id_to_label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Convert true_labels and true_predictions to binary arrays
    mlb = MultiLabelBinarizer()
    true_labels = mlb.fit_transform(true_labels)
    true_predictions = mlb.transform(true_predictions)

    results = classification_report(
        true_labels, true_predictions, zero_division=0, output_dict=True
    )
    return {
        "precision": results["weighted avg"]["precision"],
        "recall": results["weighted avg"]["recall"],
        "f1": results["weighted avg"]["f1-score"],
    }

In [ ]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2691: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss


In [36]:
from sklearn.preprocessing import MultiLabelBinarizer

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id_to_label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Convert true_labels and true_predictions to binary arrays using MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    true_labels = mlb.fit_transform(true_labels)
    true_predictions = mlb.transform(true_predictions)

    results = classification_report(
        true_labels, true_predictions, zero_division=0, output_dict=True
    )
    return {
        "precision": results["weighted avg"]["precision"],
        "recall": results["weighted avg"]["recall"],
        "f1": results["weighted avg"]["f1-score"],
    }